# 01 — CNN simple, sin filtrado ni limpieza

Primer notebook de la comparación de arquitecturas. Es intencionadamente el más simple posible: señal sin filtrar (sin `preprocess_ecg`), sin balanceo de clases, sin optimización de umbral, y una red con un único bloque convolucional. Sirve de suelo de referencia frente al que medir cada mejora posterior, tanto de arquitectura como de preprocesado.

## 1. Entorno, semillas y librerías

In [2]:
import sys, os, random, hashlib, json
from pathlib import Path

import numpy as np
import pandas as pd
import ast

import wfdb
from scipy import signal as sp_signal

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

import matplotlib.pyplot as plt

print(sys.executable)
print("TensorFlow:", tf.__version__)

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


C:\Users\Usuario\anaconda3\envs\tfg\python.exe
TensorFlow: 2.16.1


## 2. Configuración general

`NOTEBOOK_NAME` y `PREP_VERSION` son las dos variables que cambian entre notebooks de esta serie: identifican la fila en la tabla comparativa y la carpeta de caché, respectivamente.

In [3]:
NOTEBOOK_NAME = "01_cnn_simple_sin_filtrado"
PREP_VERSION  = "sin_filtrado"     # mismo valor en cualquier notebook que tampoco filtre la señal

DATA_DIR  = Path(r"C:\Users\Usuario\Desktop\TFG\ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3")
CACHE_DIR = DATA_DIR / "cache_v2" / PREP_VERSION
CACHE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_DIR = Path("./outputs/models"); MODEL_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR   = Path("./outputs/figures"); FIG_DIR.mkdir(parents=True, exist_ok=True)
COMPARISON_CSV = MODEL_DIR / "model_comparison.csv"   # compartido por TODOS los notebooks de esta serie

FS = 100
SUPER_CLASSES = ["NORM", "MI", "STTC", "HYP", "CD"]
CLASS_MAP_HUMAN = {
    "NORM": "Normal", "MI": "Infarto", "STTC": "Isquemia/ST-T",
    "HYP": "Hipertrofia", "CD": "Conducción (bloqueos)",
}

STFT_NPERSEG  = 64
STFT_NOVERLAP = 48
SPEC_SHAPE    = (33, 64)


## 3. Carga de metadatos y etiquetas multi-etiqueta

In [4]:
df  = pd.read_csv(DATA_DIR / "ptbxl_database.csv", index_col="ecg_id")
scp = pd.read_csv(DATA_DIR / "scp_statements.csv", index_col=0)
df.columns  = df.columns.str.strip()
scp.columns = scp.columns.str.strip()

scp_super = (scp[scp["diagnostic_class"].notna()][["diagnostic_class"]]
             .rename(columns={"diagnostic_class": "superclass"}))

def extract_superclasses(scp_codes_str):
    """Convierte el string tipo diccionario de scp_codes en la lista de superclases presentes."""
    if pd.isna(scp_codes_str):
        return []
    try:
        d = ast.literal_eval(scp_codes_str)
        if not isinstance(d, dict):
            return []
    except Exception:
        return []
    return list({scp_super.loc[k, "superclass"] for k in d.keys() if k in scp_super.index})

def multilabel_vector(classes):
    """Convierte la lista de superclases en un vector binario de tamaño 5."""
    s = set(classes)
    return np.array([1 if c in s else 0 for c in SUPER_CLASSES], dtype=np.int32)

df["superclasses"] = df["scp_codes"].apply(extract_superclasses)
df = df[df["superclasses"].apply(len) > 0].reset_index()
df["target_vec"] = df["superclasses"].apply(multilabel_vector)


## 4. Split por paciente

In [5]:
train_df = df[df["strat_fold"].isin(range(1, 9))].reset_index(drop=True)
val_df   = df[df["strat_fold"] == 9].reset_index(drop=True)
test_df  = df[df["strat_fold"] == 10].reset_index(drop=True)

assert set(train_df.patient_id) & set(val_df.patient_id) == set()
assert set(train_df.patient_id) & set(test_df.patient_id) == set()
assert set(val_df.patient_id)  & set(test_df.patient_id)  == set()

print("Tamaños:", len(train_df), len(val_df), len(test_df))


Tamaños: 17084 2146 2158


## 5. Extracción de características: espectrograma STFT


In [6]:
def lead_to_spectrogram(lead_signal, fs=FS, nperseg=STFT_NPERSEG, noverlap=STFT_NOVERLAP, out_shape=SPEC_SHAPE):
    """Convierte una derivación 1D en su espectrograma STFT normalizado, con forma fija (F, T)."""
    _, _, Zxx = sp_signal.stft(lead_signal, fs=fs, nperseg=nperseg, noverlap=noverlap)
    mag = np.log1p(np.abs(Zxx))

    F, T = out_shape
    mag = mag[:F, :T]
    if mag.shape != (F, T):
        pad = np.zeros((F, T), dtype=np.float32)
        pad[:mag.shape[0], :mag.shape[1]] = mag
        mag = pad

    mn, mx = mag.min(), mag.max()
    if mx - mn > 1e-8:
        mag = (mag - mn) / (mx - mn)
    return mag.astype(np.float32)

def ecg_to_tensor(signal_2d):
    """signal_2d: (n_muestras, 12) -> tensor (F, T, 12)."""
    specs = [lead_to_spectrogram(signal_2d[:, i]) for i in range(signal_2d.shape[1])]
    return np.stack(specs, axis=-1)


## 6. Generación del dataset con caché en disco

**Este notebook no llama a `preprocess_ecg` en ningún momento** — la señal cruda de `wfdb.rdsamp` se convierte directamente en espectrograma. Es la única diferencia real respecto a un notebook "con_filtrado": basta con insertar aquí la limpieza para convertir este notebook en su pareja filtrada.

In [7]:
def _cache_path(rel_path):
    h = hashlib.md5(rel_path.encode()).hexdigest()
    return CACHE_DIR / f"{h}.npy"

def get_tensor_cached(rel_path):
    cp = _cache_path(rel_path)
    if cp.exists():
        return np.load(cp)
    raw, _ = wfdb.rdsamp(str(DATA_DIR / rel_path))
    tensor = ecg_to_tensor(raw.astype(np.float32))
    np.save(cp, tensor)
    return tensor

def build_split(df_split):
    Xs, Ys = [], []
    for _, row in df_split.iterrows():
        t = get_tensor_cached(row["filename_lr"])
        Xs.append(t)
        Ys.append(row["target_vec"])
    return np.stack(Xs).astype(np.float32), np.stack(Ys).astype(np.float32)

X_train, Y_train = build_split(train_df)
X_val,   Y_val   = build_split(val_df)
X_test,  Y_test  = build_split(test_df)

print("Formas:", X_train.shape, X_val.shape, X_test.shape)


Formas: (17084, 33, 64, 12) (2146, 33, 64, 12) (2158, 33, 64, 12)


## 7. Arquitectura: CNN mínima

Un único bloque convolucional (32 filtros) seguido directamente de la clasificación. Nada de regularización L2, dropout mínimo solo para evitar un colapso total. Es deliberadamente pequeña: el objetivo de este notebook es el suelo de la comparación, no un buen modelo.

In [8]:
def build_model_simple(input_shape, n_classes):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(inputs)
    x = layers.MaxPooling2D(2)(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(n_classes, activation="sigmoid")(x)
    return models.Model(inputs, outputs)

model = build_model_simple(X_train.shape[1:], len(SUPER_CLASSES))
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="binary_crossentropy",
              metrics=[tf.keras.metrics.AUC(name="auc", multi_label=True)])
model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)             │ (None, 33, 64, 12)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d (Conv2D)                      │ (None, 33, 64, 32)          │           3,488 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 16, 32, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 32)                  │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 5)                   │             165 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 3,653 (14.27 KB)

 Trainable params: 3,653 (14.27 KB)

 Non-trainable params: 0 (0.00 B)

## 8. Entrenamiento

Sin `pos_weight` ni focal loss: `binary_crossentropy` simple, umbral de decisión fijo en 0.5.

In [9]:
cbs = [
    callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=8, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor="val_auc", mode="max", factor=0.5, patience=4, min_lr=1e-6),
    callbacks.ModelCheckpoint(str(MODEL_DIR / f"{NOTEBOOK_NAME}.keras"), monitor="val_auc",
                               mode="max", save_best_only=True),
]

history = model.fit(
    X_train, Y_train,
    validation_data=(X_val, Y_val),
    epochs=60,
    batch_size=32,
    callbacks=cbs,
    verbose=1,
)


Epoch 1/60
534/534 ━━━━━━━━━━━━━━━━━━━━ 13s 21ms/step - auc: 0.5426 - loss: 0.5502 - val_auc: 0.6017 - val_loss: 0.5320 - learning_rate: 0.0010
Epoch 2/60
534/534 ━━━━━━━━━━━━━━━━━━━━ 11s 21ms/step - auc: 0.6097 - loss: 0.5287 - val_auc: 0.6353 - val_loss: 0.5216 - learning_rate: 0.0010
Epoch 3/60
534/534 ━━━━━━━━━━━━━━━━━━━━ 13s 25ms/step - auc: 0.6422 - loss: 0.5194 - val_auc: 0.6569 - val_loss: 0.5146 - learning_rate: 0.0010
Epoch 4/60
534/534 ━━━━━━━━━━━━━━━━━━━━ 14s 25ms/step - auc: 0.6621 - loss: 0.5115 - val_auc: 0.6739 - val_loss: 0.5083 - learning_rate: 0.0010
Epoch 5/60
534/534 ━━━━━━━━━━━━━━━━━━━━ 14s 25ms/step - auc: 0.6765 - loss: 0.5051 - val_auc: 0.6901 - val_loss: 0.5022 - learning_rate: 0.0010
Epoch 6/60
534/534 ━━━━━━━━━━━━━━━━━━━━ 14s 27ms/step - auc: 0.6929 - loss: 0.4983 - val_auc: 0.7057 - val_loss: 0.4955 - learning_rate: 0.0010
Epoch 7/60
534/534 ━━━━━━━━━━━━━━━━━━━━ 11s 20ms/step - auc: 0.7059 - loss: 0.4918 - val_auc: 0.7189 - val_loss: 0.4892 - learning_rate:

## 9. Evaluación sobre el conjunto de test

In [10]:
def compute_metrics(Y_true, Y_pred, Y_prob):
    """Accuracy, Precision, Recall, F1 y MAE por clase, más la fila Macro (media simple)."""
    rows = []
    for i, c in enumerate(SUPER_CLASSES):
        yt, yp, ypr = Y_true[:, i], Y_pred[:, i], Y_prob[:, i]
        tp = np.sum((yt == 1) & (yp == 1))
        tn = np.sum((yt == 0) & (yp == 0))
        fp = np.sum((yt == 0) & (yp == 1))
        fn = np.sum((yt == 1) & (yp == 0))
        acc  = (tp + tn) / len(yt)
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
        mae  = np.mean(np.abs(ypr - yt))
        rows.append([CLASS_MAP_HUMAN[c], acc, prec, rec, f1, mae])

    metrics_df = pd.DataFrame(rows, columns=["Clase", "Accuracy", "Precision", "Recall", "F1", "MAE"])
    macro = metrics_df[["Accuracy", "Precision", "Recall", "F1", "MAE"]].mean()
    metrics_df.loc[len(metrics_df)] = ["Macro"] + macro.tolist()
    return metrics_df

test_probs = model.predict(X_test, verbose=0)
test_pred  = (test_probs >= 0.5).astype(int)

metrics_df = compute_metrics(Y_test, test_pred, test_probs)
metrics_df


,Clase,Accuracy,Precision,Recall,F1,MAE
0,Normal,0.821131,0.760614,0.874351,0.813527,0.282107
1,Infarto,0.773401,0.680473,0.209091,0.319889,0.289750
2,Isquemia/ST-T,0.817424,0.761317,0.355086,0.484293,0.251115
3,Hipertrofia,0.879518,0.750000,0.011450,0.022556,0.180824
4,Conducción (bloqueos),0.823911,0.849398,0.284274,0.425982,0.268367
5,Macro,0.823077,0.760360,0.346851,0.413249,0.254433


## 10. Registro en la tabla comparativa común

In [14]:
macro_row = metrics_df[metrics_df["Clase"] == "Macro"].drop(columns="Clase").iloc[0]
result_row = pd.DataFrame([{
    "Notebook": NOTEBOOK_NAME,
    "Preprocesado": PREP_VERSION,
    "Parametros_entrenables": model.count_params(),
    **macro_row.to_dict(),
}])

if COMPARISON_CSV.exists():
    comparison = pd.read_csv(COMPARISON_CSV)
    comparison = comparison[comparison["Notebook"] != NOTEBOOK_NAME]
    comparison = pd.concat([comparison, result_row], ignore_index=True)
else:
    comparison = result_row

comparison.to_csv(COMPARISON_CSV, index=False)
comparison


,Notebook,Preprocesado,Parametros_entrenables,Accuracy,Precision,Recall,F1,MAE
0,02_cnn_simple_con_filtrado,con_filtrado,3653,0.829472,0.733107,0.387682,0.457371,0.241108
1,03_cnn_profunda_con_filtrado,con_filtrado,105317,0.854310,0.733758,0.591347,0.630620,0.178831
2,04_cnn_profunda_balanceo,con_filtrado,105317,0.810936,0.577661,0.792424,0.656731,0.286431
3,05_cnn_profunda_balanceo_umbral,con_filtrado,105317,0.840871,0.637769,0.722755,0.672034,0.322418
4,01_cnn_simple_sin_filtrado,sin_filtrado,3653,0.823077,0.760360,0.346851,0.413249,0.254433


In [15]:
PER_CLASS_CSV = MODEL_DIR / "per_class_comparison.csv"

per_class_rows = metrics_df.copy()
per_class_rows.insert(0, "Notebook", NOTEBOOK_NAME)

if PER_CLASS_CSV.exists():
    per_class_all = pd.read_csv(PER_CLASS_CSV)
    per_class_all = per_class_all[per_class_all["Notebook"] != NOTEBOOK_NAME]
    per_class_all = pd.concat([per_class_all, per_class_rows], ignore_index=True)
else:
    per_class_all = per_class_rows

per_class_all.to_csv(PER_CLASS_CSV, index=False)
per_class_rows

,Notebook,Clase,Accuracy,Precision,Recall,F1,MAE
0,01_cnn_simple_sin_filtrado,Normal,0.821131,0.760614,0.874351,0.813527,0.282107
1,01_cnn_simple_sin_filtrado,Infarto,0.773401,0.680473,0.209091,0.319889,0.289750
2,01_cnn_simple_sin_filtrado,Isquemia/ST-T,0.817424,0.761317,0.355086,0.484293,0.251115
3,01_cnn_simple_sin_filtrado,Hipertrofia,0.879518,0.750000,0.011450,0.022556,0.180824
4,01_cnn_simple_sin_filtrado,Conducción (bloqueos),0.823911,0.849398,0.284274,0.425982,0.268367
5,01_cnn_simple_sin_filtrado,Macro,0.823077,0.760360,0.346851,0.413249,0.254433
